# Module 2: AgentCore Runtime — Your First Deployed Agent

![Overview](../shared/img/02.drawio.png)

In this module, you will deploy Aria as a live agent running on **Amazon Bedrock AgentCore Runtime**.

## What you'll learn

- **AgentCore Runtime** — the managed compute layer that hosts, scales, and versions your agents
- **How agent code is packaged and deployed** — from a local directory to a running container in the cloud
- **The streaming invocation model** — how requests flow from client to agent and back as a stream of events

By the end of this module Aria will be live and responding to prompts, though she won't yet have any tools or memory.

---
## Catch up

The cell below ensures all prerequisites and prior module outputs are in place. If you are jumping into this module directly, it will run any setup needed from earlier modules.

In [ ]:
import sys; sys.path.insert(0, '..')
from shared.ensure_ready import ensure_ready

config = ensure_ready("02")

---
## Explore the agent code

The agent lives in `agent/main.py`. Before deploying it, let's understand the key pieces.

### BedrockAgentCoreApp

Every AgentCore agent starts with a **`BedrockAgentCoreApp`** instance. This is the framework's entry point — it handles the HTTP lifecycle, health checks, and communication with the Runtime control plane.

### The `@app.entrypoint` decorator

You register your agent logic with `@app.entrypoint`. The decorated function receives:
- **`payload`** — the incoming request (contains the user message, session info, etc.)
- **`context`** — optional metadata about the invocation

### Async streaming pattern

The entrypoint is an **async generator**. Instead of returning a single response, it **yields events** as the agent thinks and responds. This gives the caller real-time streaming output — the same experience you get in the Bedrock console.

### What does Runtime actually do?

When you deploy to AgentCore Runtime, the service:
1. **Packages** your code into a container image
2. **Provisions** a dedicated **microVM for each session** — complete isolation of CPU, memory, and filesystem
3. **Scales** automatically based on traffic — including scaling to zero when idle
4. **Versions** each deployment so you can roll back if needed

### Session isolation — the key insight

This is one of Runtime's most important features: **each session gets its own microVM**. Your agent code only ever handles one session at a time. This means:

- **No multi-session management** — you don't need a session-keyed cache or routing logic. Just create your Agent object and reuse it across invocations.
- **Conversation history is automatic** — the Strands Agent keeps its message list in memory, and since the microVM persists between invocations (up to 8 hours), follow-up messages in the same session see the full conversation history.
- **True isolation** — one user's session can never access another user's data. After a session ends, the entire microVM is terminated and memory is sanitized.

This is why the agent code is so simple — Runtime handles the hard infrastructure problems so your agent code can focus on logic.

To see the full agent code, open [agent/main.py](agent/main.py) in a new tab.

The agent's Python dependencies are listed in [agent/requirements.txt](agent/requirements.txt).

---
## Deploy the agent

Deployment follows a straightforward pipeline:

1. **CodeZip packaging** — your agent directory is zipped into a deployment artifact
2. **S3 upload** — the artifact is uploaded to the workshop's S3 bucket
3. **Runtime creation** — AgentCore pulls the artifact, builds a container, and starts the agent
4. **READY state** — the agent is live and accepting invocations

The `deploy_agent` helper handles all of this in a single call.

> **Other deployment options:** In this workshop we use a Python helper that calls the AgentCore control-plane API directly. You can also deploy agents using the [AgentCore CLI](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/agentcore-get-started-toolkit.html), [AWS CDK](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/agents-tools-runtime.html), or the [AWS Management Console](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/agents-tools-runtime.html). See the [AgentCore Runtime documentation](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/agents-tools-runtime.html) for details on each approach.

In [ ]:
import sys; sys.path.insert(0, '..')
from shared import deploy_agent

result = deploy_agent.deploy(
    agent_dir="agent",
    runtime_name="aria_agent",
    clean_start=True,
)

runtime_arn = result["runtime_arn"]
print(f"\nRuntime ARN: {runtime_arn}")

---
## Enable Tracing

Before invoking the agent, enable **Tracing** for the Runtime so that invocation traces flow to CloudWatch and X-Ray. This populates the AgentCore Observability dashboard you will explore in Module 7.

1. Open the **Amazon Bedrock AgentCore** console and navigate to **Runtimes**
2. Select the **aria_agent** runtime
3. Scroll down to the **Tracing** section and click **Edit**

![Tracing section](../shared/img/tracing-01.png)

4. Toggle **Enable** on and click **Save**

![Enable tracing](../shared/img/tracing-02.png)

> **Note:** Tracing must be enabled manually per resource. You will repeat this step for Memory and Gateway in later modules.

---
## Invoke the agent

Aria is live. Let's call the deployed agent using the boto3 `bedrock-agentcore` data-plane client. This is the raw API call — no helper functions.

You send a JSON payload with a `prompt` key, and the agent streams back a response as **Server-Sent Events (SSE)**. Let's start by looking at the raw response to understand the event format.

In [ ]:
import boto3
import json
import uuid

client = boto3.client("bedrock-agentcore", region_name="us-east-1")

response = client.invoke_agent_runtime(
    agentRuntimeArn=runtime_arn,
    runtimeSessionId=str(uuid.uuid4()),
    contentType="application/json",
    accept="text/event-stream",
    payload=json.dumps({"prompt": "Hello! What can you do?"}).encode("utf-8"),
)

for line in response["response"].iter_lines():
    if line:
        print(line.decode("utf-8"), flush=True)

### Understanding the response format

The raw output is a stream of **Server-Sent Events (SSE)**. Each line starting with `data: ` contains a JSON object. The agent's text lives inside a nested path:

```
event → contentBlockDelta → delta → text
```

Let's parse this properly to extract just the agent's text:

In [ ]:
# Invoke again, this time parsing the SSE stream for clean output
import sys; sys.path.insert(0, '..')
from shared import utils

response = client.invoke_agent_runtime(
    agentRuntimeArn=runtime_arn,
    runtimeSessionId=str(uuid.uuid4()),
    payload=json.dumps({"prompt": "Hello! What can you do?"}).encode(),
)

utils.stream_sse_response(response["response"])

---
## Chat with Aria interactively

Now that you understand the invocation model, let's talk to Aria using an interactive CLI chat client. This is a shared tool you will use throughout the workshop for hands-on testing.

Open a **terminal** in your environment:
- In VS Code: **Terminal > New Terminal** (or press `` Ctrl+` ``)
- Navigate to this module's directory

Then run:

```bash
cd /workshop/02-runtime
python ../shared/chat.py
```

Try these prompts to explore what Aria can and cannot do:

1. `Hello! What can you do?`
2. `What is the square root of 7,293,461?` — she may hallucinate a number rather than calculate it
3. `What is today's weather in Seattle?` — she has no internet access
4. `My name is Alex` — then in the same session: `What's my name?` — she remembers within a session (microVM persistence)
5. Type `new` to start a fresh session, then ask `What's my name?` — she won't remember (no long-term memory yet)
6. Type `quit` to exit

> **Tip:** The chat client uses the same `invoke_agent_runtime` API call you saw above. It reads config from `shared/.config/` automatically. Later in the workshop, when we add authentication, you can run it with `--auth` to use OAuth/JWT.

If you tried the prompts above, you will have noticed:

- **No computation** — asking for a square root may produce a hallucinated number rather than a precise calculation.
- **No live data** — she cannot look up today's weather, stock prices, or news.
- **No persistent memory** — within a session she remembers (thanks to the microVM), but across sessions she starts fresh every time.

---
## What's next

Aria is deployed and responding, but she is limited to what the model already knows. She can't run code, browse the web, or remember past conversations.

In **Module 3: Managed Tools**, we will give Aria two powerful capabilities:
- **Code Interpreter** — sandboxed Python execution for calculations, data analysis, and chart generation
- **Browser Tool** — headless Chrome for live web browsing and content extraction

These are managed services provided by AgentCore — no infrastructure for you to maintain.

---
## Record progress

In [ ]:
import sys; sys.path.insert(0, '..')
from shared import progress

progress.show("02")

---

**Next up: [Module 3 -- Supercharge Aria with Tools](../03-tools/notebook.ipynb)**